In [5]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU: []


In [9]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

project_dir = r"D:\vision_task_files"
local_data_dir = r"D:\archive\EuroSAT"

train_df = pd.read_csv(os.path.join(project_dir, "train_split.csv"))
val_df = pd.read_csv(os.path.join(project_dir, "val_split.csv"))
test_df = pd.read_csv(os.path.join(project_dir, "test_split.csv"))

kaggle_data_dir = "/kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSAT"

train_df["image_path"] = train_df["image_path"].str.replace(kaggle_data_dir, local_data_dir, regex=False)
val_df["image_path"] = val_df["image_path"].str.replace(kaggle_data_dir, local_data_dir, regex=False)
test_df["image_path"] = test_df["image_path"].str.replace(kaggle_data_dir, local_data_dir, regex=False)

print(train_df.head())
print("Path exists:", os.path.exists(train_df["image_path"].iloc[0]))

                                          image_path        label
0  D:\archive\EuroSAT/Residential/Residential_798...  Residential
1        D:\archive\EuroSAT/SeaLake/SeaLake_1177.jpg      SeaLake
2  D:\archive\EuroSAT/Residential/Residential_216...  Residential
3    D:\archive\EuroSAT/AnnualCrop/AnnualCrop_61.jpg   AnnualCrop
4           D:\archive\EuroSAT/Forest/Forest_463.jpg       Forest
Path exists: True


In [11]:
class_names = sorted(train_df["label"].unique())
label_to_index = {label: idx for idx, label in enumerate(class_names)}
NUM_CLASSES = len(class_names)

train_paths = train_df["image_path"].values
val_paths = val_df["image_path"].values
test_paths = test_df["image_path"].values

train_labels = train_df["label"].map(label_to_index).values
val_labels = val_df["label"].map(label_to_index).values
test_labels = test_df["label"].map(label_to_index).values

print(class_names)
print("Number of classes:", NUM_CLASSES)

['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Number of classes: 10


In [13]:
IMG_SIZE_TL = (96, 96)
BATCH_SIZE_TL = 32

def load_image_transfer(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE_TL)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds_tl = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds_tl = train_ds_tl.shuffle(buffer_size=len(train_paths), seed=42)
train_ds_tl = train_ds_tl.map(load_image_transfer, num_parallel_calls=AUTOTUNE)
train_ds_tl = train_ds_tl.batch(BATCH_SIZE_TL).prefetch(AUTOTUNE)

val_ds_tl = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds_tl = val_ds_tl.map(load_image_transfer, num_parallel_calls=AUTOTUNE)
val_ds_tl = val_ds_tl.batch(BATCH_SIZE_TL).prefetch(AUTOTUNE)

test_ds_tl = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds_tl = test_ds_tl.map(load_image_transfer, num_parallel_calls=AUTOTUNE)
test_ds_tl = test_ds_tl.batch(BATCH_SIZE_TL).prefetch(AUTOTUNE)

print("Transfer learning datasets ready.")

Transfer learning datasets ready.


In [15]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3),
    alpha=0.35
)

base_model.trainable = False

transfer_model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")
])

transfer_model.summary()

2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 2s 1us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_0.35_96             │ (None, 3, 3, 1280)     │       410,208 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 575,466 (2.20 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 410,208 (1.56 MB)

In [17]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

history_transfer = transfer_model.fit(
    train_ds_tl,
    validation_data=val_ds_tl,
    epochs=3
)

end_time = time.time()

transfer_training_time = end_time - start_time
print("Training time:", transfer_training_time)

Epoch 1/3
591/591 ━━━━━━━━━━━━━━━━━━━━ 124s 193ms/step - accuracy: 0.7333 - loss: 0.8196 - val_accuracy: 0.8731 - val_loss: 0.3825
Epoch 2/3
591/591 ━━━━━━━━━━━━━━━━━━━━ 90s 105ms/step - accuracy: 0.8630 - loss: 0.4175 - val_accuracy: 0.8963 - val_loss: 0.3124
Epoch 3/3
591/591 ━━━━━━━━━━━━━━━━━━━━ 67s 113ms/step - accuracy: 0.8836 - loss: 0.3486 - val_accuracy: 0.9012 - val_loss: 0.2799
Training time: 280.58631896972656


In [19]:
test_loss_transfer, test_acc_transfer = transfer_model.evaluate(test_ds_tl)

print("Transfer Learning Test Loss:", test_loss_transfer)
print("Transfer Learning Test Accuracy:", test_acc_transfer)

127/127 ━━━━━━━━━━━━━━━━━━━━ 20s 155ms/step - accuracy: 0.9074 - loss: 0.2793
Transfer Learning Test Loss: 0.27930957078933716
Transfer Learning Test Accuracy: 0.9074074029922485
